# Train / Validation Label Statistics

Per-split statistics over the raw RHE labels (per timestep, before windowing / trimming) for the
missions in `TRAIN_MISSIONS` and `VALID_MISSIONS` of the training config.

Computes:
- mu distribution (plots saved to the output folder, like `label_statistics_plots.ipynb`)
- fraction of low-mu samples whose mu observability is low but non-zero (i.e. the rare low-mu
  samples whose LDS loss weight gets attenuated, not zeroed, by the observability gate).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from config.paths import LABELS_ROOT, OUTPUT_ROOT
from traversability_network.config import get_cfg


def find_repo_root(start: Path) -> Path:
    rel = Path("training/traversability_network/configs/grandtour_lite.yaml")
    p = start.resolve()
    while p != p.parent:
        if (p / rel).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"could not locate {rel} above {start}")


REPO_ROOT = find_repo_root(Path.cwd())
CFG_FILE = REPO_ROOT / "training/traversability_network/configs/grandtour_lite.yaml"
cfg = get_cfg(str(CFG_FILE))

# The mu column of rhe_trav_labels.npy, and the low-mu / observability thresholds
# (defaults mirror the oversampling knobs so 'low mu' matches what training targets).
MU_COLUMN = int(cfg.MODEL.LABEL_COLUMNS[0])
LOW_MU_MAX = cfg.DATASET.OVERSAMPLE_MU_MAX
OBS_THRESH = cfg.DATASET.OVERSAMPLE_OBSERVABILITY_THRESH

graphs_root = OUTPUT_ROOT / "train_val_label_statistics"
graphs_root.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 150,
})

print(f"config: {CFG_FILE}")
print(f"labels: {LABELS_ROOT}")
print(f"low-mu threshold: mu <= {LOW_MU_MAX} | observability threshold: {OBS_THRESH}")
print(f"train missions: {cfg.DATASET.TRAIN_MISSIONS}")
print(f"valid missions: {cfg.DATASET.VALID_MISSIONS}")

config: /workspaces/master-thesis-lmertens/training/traversability_network/configs/grandtour_lite.yaml
labels: /datahome/lmertens/grand_tour_dataset/labels
low-mu threshold: mu <= 0.6 | observability threshold: 0.6
train missions: ['ALB-1', 'ALB-2', 'CYN-1', 'HOEB-1', 'LMB-1', 'RIV-1', 'TRIM-1', 'HIL-1', 'KAEB-2', 'KAEB-3']
valid missions: ['ALB-3', 'LMB-2', 'HOEB-2', 'CYN-2']


In [2]:
def load_mission(mission: str):
    """Raw per-timestep mu and its observability for one mission."""
    rhe_dir = LABELS_ROOT / mission / "rhe"
    labels_path = rhe_dir / "rhe_trav_labels.npy"
    obs_path = rhe_dir / "rhe_mu_observability.npy"
    if not labels_path.exists():
        print(f"skipping {mission}: {labels_path} missing")
        return None
    mu = np.load(labels_path)[:, MU_COLUMN]
    mu_obs = np.load(obs_path) if obs_path.exists() else np.ones_like(mu)
    return {"mission": mission, "mu": mu, "mu_obs": mu_obs}


def load_split(missions: list):
    stats = [s for s in (load_mission(m) for m in missions) if s is not None]
    mu = np.concatenate([s["mu"] for s in stats]) if stats else np.array([])
    mu_obs = np.concatenate([s["mu_obs"] for s in stats]) if stats else np.array([])
    return {"per_mission": stats, "mu": mu, "mu_obs": mu_obs}


splits = {
    "train": load_split(cfg.DATASET.TRAIN_MISSIONS),
    "valid": load_split(cfg.DATASET.VALID_MISSIONS),
}
for name, s in splits.items():
    print(
        f"{name}: {s['mu'].size} label timesteps from {len(s['per_mission'])} missions"
    )

train: 41254 label timesteps from 10 missions
valid: 11544 label timesteps from 4 missions


In [3]:
def split_summary(mu, mu_obs):
    n = mu.size
    low = mu <= LOW_MU_MAX
    n_low = int(low.sum())
    obs_low = mu_obs[low]
    n_obs_zero = int((obs_low <= 0).sum())
    n_obs_low_nonzero = int(((obs_low > 0) & (obs_low < OBS_THRESH)).sum())
    n_obs_high = int((obs_low >= OBS_THRESH).sum())
    pct = lambda a, b: 100.0 * a / b if b else np.nan
    return {
        "n_samples": n,
        "mu_mean": float(np.mean(mu)) if n else np.nan,
        "mu_median": float(np.median(mu)) if n else np.nan,
        f"n_low_mu(<= {LOW_MU_MAX})": n_low,
        "pct_low_mu": pct(n_low, n),
        # of the low-mu samples, split by observability
        "low_mu_pct_obs_zero": pct(n_obs_zero, n_low),
        "low_mu_pct_obs_low_nonzero": pct(n_obs_low_nonzero, n_low),
        "low_mu_pct_obs_high": pct(n_obs_high, n_low),
    }


summary = pd.DataFrame({
    name: split_summary(s["mu"], s["mu_obs"]) for name, s in splits.items()
}).T
print(f"observability buckets computed over low-mu samples (mu <= {LOW_MU_MAX}):")
print("  obs_zero          : observability == 0  (masked out of the loss entirely)")
print(
    f"  obs_low_nonzero   : 0 < observability < {OBS_THRESH}  (LDS weight attenuated, not zeroed)"
)
print(
    f"  obs_high          : observability >= {OBS_THRESH}  (survives the oversample threshold)"
)
summary.round(2)

observability buckets computed over low-mu samples (mu <= 0.6):
  obs_zero          : observability == 0  (masked out of the loss entirely)
  obs_low_nonzero   : 0 < observability < 0.6  (LDS weight attenuated, not zeroed)
  obs_high          : observability >= 0.6  (survives the oversample threshold)


,n_samples,mu_mean,mu_median,n_low_mu(<= 0.6),pct_low_mu,low_mu_pct_obs_zero,low_mu_pct_obs_low_nonzero,low_mu_pct_obs_high
train,41254.0,0.84,0.87,1700.0,4.12,15.76,4.94,79.29
valid,11544.0,0.85,0.86,228.0,1.98,1.75,6.14,92.11


In [4]:
# Per-mission breakdown of the same low-mu observability split.
rows = []
for name, s in splits.items():
    for m in s["per_mission"]:
        r = split_summary(m["mu"], m["mu_obs"])
        r = {"split": name, "mission": m["mission"], **r}
        rows.append(r)
per_mission_df = pd.DataFrame(rows)
per_mission_df.round(2)

,split,mission,n_samples,mu_mean,mu_median,n_low_mu(<= 0.6),pct_low_mu,low_mu_pct_obs_zero,low_mu_pct_obs_low_nonzero,low_mu_pct_obs_high
0,train,ALB-1,2405,0.82,0.83,62,2.58,0.00,0.00,100.00
1,train,ALB-2,4599,0.75,0.81,770,16.74,4.42,5.45,90.13
2,train,CYN-1,4464,0.89,0.89,0,0.00,NaN,NaN,NaN
3,train,HOEB-1,4742,0.86,0.87,4,0.08,0.00,0.00,100.00
4,train,LMB-1,4155,0.76,0.81,400,9.63,38.50,5.50,56.00
5,train,RIV-1,4708,0.78,0.82,448,9.52,17.86,4.46,77.68
6,train,TRIM-1,4484,0.89,0.89,0,0.00,NaN,NaN,NaN
7,train,HIL-1,4050,0.89,0.90,0,0.00,NaN,NaN,NaN
8,train,KAEB-2,3218,0.85,0.86,0,0.00,NaN,NaN,NaN
9,train,KAEB-3,4429,0.88,0.89,16,0.36,0.00,0.00,100.00


In [14]:
# sum over n_samples, mean over the other metrics, grouped by split
per_split_df = per_mission_df.groupby("split").agg(
    n_samples=("n_samples", "sum"),
    mu_mean=("mu_mean", "mean"),
    mu_median=("mu_median", "mean"),
    n_low_mu=("n_low_mu(<= 0.6)", "sum"),
    pct_low_mu=("pct_low_mu", "mean"),
    low_mu_pct_obs_zero=("low_mu_pct_obs_zero", "mean"),
    low_mu_pct_obs_low_nonzero=("low_mu_pct_obs_low_nonzero", "mean"),
    low_mu_pct_obs_high=("low_mu_pct_obs_high", "mean"),
)

per_split_df.round(2)

,n_samples,mu_mean,mu_median,n_low_mu,pct_low_mu,low_mu_pct_obs_zero,low_mu_pct_obs_low_nonzero,low_mu_pct_obs_high
split,,,,,,,,
train,41254,0.84,0.86,1700,3.89,10.13,2.57,87.30
valid,11544,0.84,0.85,228,3.01,1.25,4.38,94.38


In [ ]:
BINS = np.linspace(0.0, 1.0, 41)
COLORS = {"train": "tab:blue", "valid": "tab:orange"}


def add_mean_median(ax, values, color):
    ax.axvline(
        float(np.mean(values)), color=color, linestyle="--", linewidth=1.5, label="mean"
    )
    ax.axvline(
        float(np.median(values)),
        color=color,
        linestyle=":",
        linewidth=1.8,
        label="median",
    )


def save_figure(fig, filename):
    path = graphs_root / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {path}")


def plot_mu_distribution(name, mu):
    fig, ax = plt.subplots()
    ax.hist(
        mu, bins=BINS, density=False, color=COLORS[name], alpha=0.75, edgecolor="white"
    )
    add_mean_median(ax, mu, "black")
    ax.axvline(
        LOW_MU_MAX,
        color="tab:red",
        linewidth=1.5,
        label=f"low-mu threshold ({LOW_MU_MAX})",
    )
    ax.set_title(f"mu distribution ({name} set)")
    ax.set_xlabel("mu")
    ax.set_ylabel("Count")
    ax.set_xlim(0.0, 1.0)
    ax.legend(loc="best")
    save_figure(fig, f"mu_distribution_{name}.pdf")


for name, s in splits.items():
    if s["mu"].size:
        plot_mu_distribution(name, s["mu"])

# Overlay train vs valid
fig, ax = plt.subplots()
for name, s in splits.items():
    if not s["mu"].size:
        continue
    ax.hist(
        s["mu"],
        bins=BINS,
        density=False,
        color=COLORS[name],
        alpha=0.5,
        edgecolor="white",
        label=f"{name} (n={s['mu'].size})",
    )
ax.axvline(
    LOW_MU_MAX, color="tab:red", linewidth=1.5, label=f"low-mu threshold ({LOW_MU_MAX})"
)
ax.set_title("mu distribution: train vs valid")
ax.set_xlabel("mu")
ax.set_ylabel("Count")
ax.set_xlim(0.0, 1.0)
ax.legend(loc="best")
save_figure(fig, "mu_distribution_train_vs_valid.pdf")

Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_train.pdf


Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_valid.pdf


Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_train_vs_valid.pdf


## Alternative ID candidate: urban / built / construction / indoor

Same mu histogram + low-mu / observability breakdown, but for the U (urban/campus) and
B (built industrial/construction) clusters from `documentation/environment_properties.md`,
to check whether a built-environment ID would give a richer low-mu tail than forest-only.


In [ ]:
# Candidate groups (families expanded to runs; kept only if labels exist on disk).
URBAN = ['GRI-1', 'LEICA-1', 'LEICA-2', 'ETH-1', 'ETH-2', 'ETH-3', 'LEE-1', 'HAUS-1']
BUILT = ['ARC-1', 'ARC-2', 'ARC-3', 'ARC-4', 'ARC-5', 'ARC-6', 'ARC-7',
         'CON-1', 'CON-2', 'CON-3', 'CON-4']

candidate_missions = {
    'urban': URBAN,
    'built': BUILT,
    'urban_built': URBAN + BUILT,
}

def keep_existing(missions):
    return [m for m in missions if (LABELS_ROOT / m / 'rhe' / 'rhe_trav_labels.npy').exists()]

candidates = {name: load_split(keep_existing(ms)) for name, ms in candidate_missions.items()}
for name, s in candidates.items():
    print(f'{name}: {s["mu"].size} label timesteps from {len(s["per_mission"])} missions')


urban: 28732 label timesteps from 8 missions
built: 44640 label timesteps from 11 missions
urban_built: 73372 label timesteps from 19 missions


In [ ]:
# Compare against the current forest ID (train + valid pooled) as reference.
forest_id = {
    'mu': np.concatenate([splits['train']['mu'], splits['valid']['mu']]),
    'mu_obs': np.concatenate([splits['train']['mu_obs'], splits['valid']['mu_obs']]),
}
compare = {'forest_id(train+valid)': forest_id, **candidates}
alt_summary = pd.DataFrame(
    {name: split_summary(s['mu'], s['mu_obs']) for name, s in compare.items()}
).T
alt_summary.round(2)


,n_samples,mu_mean,mu_median,n_low_mu(<= 0.6),pct_low_mu,low_mu_pct_obs_zero,low_mu_pct_obs_low_nonzero,low_mu_pct_obs_high
forest_id(train+valid),52798.0,0.84,0.87,1928.0,3.65,14.11,5.08,80.81
urban,28732.0,0.90,0.91,94.0,0.33,0.00,2.13,97.87
built,44640.0,0.87,0.90,1078.0,2.41,14.29,5.94,79.78
urban_built,73372.0,0.88,0.90,1172.0,1.60,13.14,5.63,81.23


In [ ]:
# Per-mission breakdown for the candidate missions.
rows = []
for name, s in candidates.items():
    for m in s['per_mission']:
        rows.append({'group': name, 'mission': m['mission'], **split_summary(m['mu'], m['mu_obs'])})
pd.DataFrame(rows).round(2)


,group,mission,n_samples,mu_mean,mu_median,n_low_mu(<= 0.6),pct_low_mu,low_mu_pct_obs_zero,low_mu_pct_obs_low_nonzero,low_mu_pct_obs_high
0,urban,GRI-1,3950,0.91,0.91,0,0.00,NaN,NaN,NaN
1,urban,LEICA-1,2506,0.93,0.92,0,0.00,NaN,NaN,NaN
2,urban,LEICA-2,3794,0.85,0.86,74,1.95,0.00,2.70,97.30
3,urban,ETH-1,2729,0.92,0.93,0,0.00,NaN,NaN,NaN
4,urban,ETH-2,3646,0.94,0.94,0,0.00,NaN,NaN,NaN
5,urban,ETH-3,4167,0.92,0.93,2,0.05,0.00,0.00,100.00
6,urban,LEE-1,4526,0.87,0.88,8,0.18,0.00,0.00,100.00
7,urban,HAUS-1,3414,0.88,0.88,10,0.29,0.00,0.00,100.00
8,built,ARC-1,4009,0.87,0.91,202,5.04,8.91,0.99,90.10
9,built,ARC-2,3620,0.84,0.86,146,4.03,6.85,1.37,91.78


In [ ]:
CAND_COLORS = {'urban': 'tab:green', 'built': 'tab:purple', 'urban_built': 'tab:red'}

def plot_group(name, mu, color):
    fig, ax = plt.subplots()
    ax.hist(mu, bins=BINS, density=True, color=color, alpha=0.75, edgecolor='white')
    add_mean_median(ax, mu, 'black')
    ax.axvline(LOW_MU_MAX, color='tab:red', linewidth=1.5, label=f'low-mu threshold ({LOW_MU_MAX})')
    ax.set_title(f'mu distribution ({name})')
    ax.set_xlabel('mu')
    ax.set_ylabel('Density')
    ax.set_xlim(0.0, 1.0)
    ax.legend(loc='best')
    save_figure(fig, f'mu_distribution_{name}.pdf')

for name, s in candidates.items():
    if s['mu'].size:
        plot_group(name, s['mu'], CAND_COLORS[name])

# Overlay: forest ID vs urban_built candidate
fig, ax = plt.subplots()
for label, mu, color in [
    ('forest_id (train+valid)', forest_id['mu'], 'tab:blue'),
    ('urban_built', candidates['urban_built']['mu'], 'tab:red'),
]:
    ax.hist(mu, bins=BINS, density=True, color=color, alpha=0.5, edgecolor='white',
            label=f'{label} (n={mu.size})')
ax.axvline(LOW_MU_MAX, color='black', linewidth=1.5, label=f'low-mu threshold ({LOW_MU_MAX})')
ax.set_title('mu distribution: forest ID vs urban/built candidate')
ax.set_xlabel('mu')
ax.set_ylabel('Density')
ax.set_xlim(0.0, 1.0)
ax.legend(loc='best')
save_figure(fig, 'mu_distribution_forest_vs_urban_built.pdf')


Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_urban.pdf


Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_built.pdf


Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_urban_built.pdf


Saved /datahome/lmertens/grand_tour_dataset/output/train_val_label_statistics/mu_distribution_forest_vs_urban_built.pdf
